# ZuCo sentiment from classical EEG features

Classical statistics of the raw and band-filtered EEG, then a linear classifier.
Mount Drive (the `.mat` files are large and live there), point `MAT_DIR` at the
task-1 folder, then run top to bottom.

In [ ]:
# clone (or pull) the code, install deps
import os
REPO = 'zuco-eeg-classical-features'
if not os.path.exists(REPO):
    !git clone https://github.com/parmisbathaeiyan/zuco-eeg-classical-features.git $REPO
else:
    !cd $REPO && git pull --ff-only
%cd $REPO
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# folder that contains results*_SR.mat (one per subject)
MAT_DIR = '/content/drive/MyDrive/zuco/task1/Matlab_files'

In [ ]:
# 1. extract features -> features/<SUBJ>.npz  (slow; resumable, skips cached)
!python extract_features.py --mat-dir "$MAT_DIR" --out-dir features

In [ ]:
# 2. classify: subject-specific + pooled
!python run.py --features-dir features --output-dir results --classifier logreg

In [ ]:
# peek at the headline numbers and a couple of plots
import json, glob
from IPython.display import Image, display

summary = json.load(open('results/subject/summary_logreg.json'))
print('subject-specific  acc %.3f +/- %.3f   macro-F1 %.3f +/- %.3f' % (
    summary['accuracy_mean'], summary['accuracy_std'],
    summary['macro_f1_mean'], summary['macro_f1_std']))
pooled = json.load(open('results/pooled/logreg.json'))
print('pooled            acc %.3f   macro-F1 %.3f   (majority %.3f)' % (
    pooled['accuracy'], pooled['macro_f1'], pooled['majority_baseline']))

for p in ['results/plots/subject_accuracy_logreg.png',
          'results/plots/cm_pooled_logreg.png']:
    if glob.glob(p):
        display(Image(p))